In [1]:
import pandas as pd
from pathlib import Path
import glob
import duckdb
import gc
import pyarrow.parquet as pq

In [2]:
folder = Path.home() / 'Documents' / 'moltbook' / 'arctic_shift'

def carica_jsonl(pattern):
    dfs = []
    for f in sorted(folder.glob(pattern)):
        df = pd.read_json(f, lines=True)
        if df.empty:
            print(f"Vuoto, salto: {f.name}")
            continue
        # r_philosophy_comments.jsonl -> philosophy
        df['subreddit_file'] = f.stem.removeprefix('r_').rsplit('_', 1)[0]
        dfs.append(df)
        print(f"{f.name}: {len(df)} righe")
    return pd.concat(dfs, ignore_index=True)

In [3]:
df_reddit_comments = carica_jsonl('r_*_comments.jsonl')

r_philosophy_comments.jsonl: 21318 righe
r_physics_comments.jsonl: 56326 righe


In [4]:
df_reddit_posts    = carica_jsonl('r_*_posts.jsonl')

r_philosophy_posts.jsonl: 2376 righe
r_physics_posts.jsonl: 4298 righe


In [5]:
df_reddit_posts["created_utc"] = pd.to_datetime(df_reddit_posts["created_utc"], unit="s", utc=True) 
df_reddit_posts = df_reddit_posts[['author','downs','id','name','num_comments','created_utc','subreddit','ups']]

In [6]:
df_reddit_posts.head()

,author,downs,id,name,num_comments,created_utc,subreddit,ups
0,PuzzleheadedSport281,0,1t0e3b6,t3_1t0e3b6,1,2026-05-01 00:35:57+00:00,philosophy,1
1,OkReport5065,0,1t0fifb,t3_1t0fifb,25,2026-05-01 01:41:06+00:00,philosophy,0
2,Silientium,0,1t0h8cx,t3_1t0h8cx,1,2026-05-01 02:59:12+00:00,philosophy,1
3,Silientium,0,1t0huhx,t3_1t0huhx,1,2026-05-01 03:28:32+00:00,philosophy,1
4,amsirm1,0,1t0j9ye,t3_1t0j9ye,1,2026-05-01 04:40:20+00:00,philosophy,1


In [7]:
# convert to datetime
df_reddit_comments["created_utc"] = pd.to_datetime(df_reddit_comments["created_utc"], unit="s", utc=True)

# remove prefix in link_id and parent_id
df = df_reddit_comments
p = df["parent_id"]

# 1) tutti i parent_id senza prefisso (tengo anche l'originale)
df["parent_id_raw"]   = p
df["parent_id_clean"] = p.str.replace(r"^t[0-9]_", "", regex=True)

# 2) solo quelli che puntano a un post (t3_)
df["parent_post_id"]    = p.where(p.str.startswith("t3_", na=False)).str[3:]

# 3) solo quelli che puntano a un commento (t1_)
df["parent_comment_id"] = p.where(p.str.startswith("t1_", na=False)).str[3:]

# link_id (il post della discussione) senza prefisso
df["link_id"] = df["link_id"].str.replace(r"^t[0-9]_", "", regex=True)
    
df_reddit_comments = df_reddit_comments[['author','created_utc','downs','id','link_id','parent_id_clean','parent_post_id','parent_comment_id','subreddit','ups']]

In [8]:
df_reddit_comments['interaction_type'] = 'comment'
df_reddit_posts['interaction_type'] = 'post'

In [9]:
df_reddit_posts = df_reddit_posts.rename(columns={'author':'agent_id'})
df_reddit_comments = df_reddit_comments.rename(columns={'author':'agent_id'})
df_reddit_posts = df_reddit_posts.rename(columns={'created_utc':'created_at'})
df_reddit_comments = df_reddit_comments.rename(columns={'created_utc':'created_at'})
df_reddit_posts = df_reddit_posts.rename(columns={'subreddit':'submolt'})
df_reddit_comments = df_reddit_comments.rename(columns={'subreddit':'submolt'})
df_reddit_comments = df_reddit_comments.rename(columns={'link_id':'post_id'})
df_reddit_comments = df_reddit_comments.rename(columns={'parent_id_clean':'parent_id'})

In [10]:
df_reddit_comments.head()

,agent_id,created_at,downs,id,post_id,parent_id,parent_post_id,parent_comment_id,submolt,ups,interaction_type
0,Rtfmlife,2026-05-01 00:19:33+00:00,0,oj8b1ff,1t07svj,1t07svj,1t07svj,NaN,philosophy,2,comment
1,AutoModerator,2026-05-01 00:35:58+00:00,0,oj8dtp4,1t0e3b6,1t0e3b6,1t0e3b6,NaN,philosophy,1,comment
2,you_are_soul,2026-05-01 00:43:02+00:00,0,oj8f0i1,1sx5cew,1sx5cew,1sx5cew,NaN,philosophy,1,comment
3,krazygreekguy,2026-05-01 00:43:16+00:00,0,oj8f1wz,1t07svj,1t07svj,1t07svj,NaN,philosophy,-2,comment
4,krazygreekguy,2026-05-01 00:43:56+00:00,0,oj8f5y6,1t07svj,oj7fht0,NaN,oj7fht0,philosophy,-6,comment


In [11]:
df_reddit_posts.head()

,agent_id,downs,id,name,num_comments,created_at,submolt,ups,interaction_type
0,PuzzleheadedSport281,0,1t0e3b6,t3_1t0e3b6,1,2026-05-01 00:35:57+00:00,philosophy,1,post
1,OkReport5065,0,1t0fifb,t3_1t0fifb,25,2026-05-01 01:41:06+00:00,philosophy,0,post
2,Silientium,0,1t0h8cx,t3_1t0h8cx,1,2026-05-01 02:59:12+00:00,philosophy,1,post
3,Silientium,0,1t0huhx,t3_1t0huhx,1,2026-05-01 03:28:32+00:00,philosophy,1,post
4,amsirm1,0,1t0j9ye,t3_1t0j9ye,1,2026-05-01 04:40:20+00:00,philosophy,1,post


In [12]:
df_reddit_comments.to_pickle('df_reddit_comments.pkl') 
df_reddit_posts.to_pickle('df_reddit_posts.pkl')

# Moltbook

In [2]:
posts_moltbook = Path.home() / 'Documents' / 'Archive_data' / 'moltbook_posts' / 'data' / 'posts'

duckdb.sql(f"""
    COPY (
        SELECT id, agent_id, submolt, comment_count, created_at, dump_date
        FROM read_parquet('{posts_moltbook}/*.parquet', union_by_name=true)
        QUALIFY row_number() OVER (PARTITION BY id ORDER BY dump_date DESC) = 1
    ) TO 'posts_ridotto_1.parquet' (FORMAT parquet)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [3]:
comments_moltbook = Path.home() / 'Documents' / 'Archive_data' / 'moltbook_comments' / 'data' / 'comments'

duckdb.sql(f"""
    COPY (
        SELECT id, post_id, agent_name, agent_id, parent_id, score, created_at, dump_date
        FROM read_parquet('{comments_moltbook}/*.parquet', union_by_name=true)
        QUALIFY row_number() OVER (PARTITION BY id ORDER BY dump_date DESC) = 1
    ) TO 'comments_ridotto_1.parquet' (FORMAT parquet)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [2]:
df_moltbook_posts = pd.read_parquet("posts_ridotto_1.parquet", dtype_backend='pyarrow')

In [14]:
df_moltbook_comments = pd.read_parquet("comments_ridotto_1.parquet", dtype_backend='pyarrow')

In [5]:
df_moltbook_comments.head()

,id,post_id,agent_name,agent_id,parent_id,score,created_at,dump_date
0,88660d9e-333a-4a7d-b545-07d38c3257a9,0f5bd8e5-9881-4b04-865f-2b5d60e39e86,HomeAI,329cda56-226f-4658-aecb-f40d930ab525,<NA>,0,2026-02-03 14:30:40.510000+00:00,2026-02-03
1,fc73fe5c-3fbd-43c7-aca3-cab2c23508a2,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,poo,d9318a43-f03a-480c-994d-e847bcf9612b,<NA>,0,2026-02-03 00:24:44.540000+00:00,2026-02-03
2,e3c0997f-3ff0-4ec0-be86-3c2e74dcafd9,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,JamesBishop,21df0603-6457-4cd6-a693-fb93da4ccc44,<NA>,0,2026-02-03 00:38:37.074000+00:00,2026-02-03
3,d31395b9-442f-48d0-8ec5-72112bdfd611,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,StratusBot,611761a2-d690-49ae-986f-33b3f5dccd36,<NA>,0,2026-02-03 01:45:12.210000+00:00,2026-02-03
4,ab8a0ec2-ad4b-4c9c-adda-aab8441779f0,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,[deleted],b6a156b8-4cf8-40aa-927f-7e4a0c758c5b,<NA>,0,2026-02-03 08:52:58.033000+00:00,2026-02-03


In [6]:
df_moltbook_posts.head()

,id,agent_id,submolt,comment_count,created_at,dump_date
0,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,606d94f2-9fde-414b-add3-d326c9ab3ccc,builds,344,2026-01-31 14:29:34.198767+00:00,2026-01-31
1,98df15b6-4f63-4782-a967-6d9aa2239cf0,37944d7b-2721-4b24-9b61-00f37bdeaa90,agents,6927,2026-01-31 12:21:30.443643+00:00,2026-01-31
2,94fc8fda-a6a9-4177-8d6b-e499adb9d675,f9658711-c03b-4c78-ae65-6cb2d33e17b7,general,48758,2026-01-31 14:00:14.933000+00:00,2026-01-31
3,75404525-5e5e-4778-ad1b-3fac43c6903d,10513ade-652d-4c8c-a410-7e686dba652c,general,31919,2026-01-31 18:11:19.470000+00:00,2026-01-31
4,c6eb531f-1ee8-428b-b1d8-41af2e9bd537,0bd24cc2-f042-4951-8400-38bd655102c6,general,5989,2026-01-31 13:12:48.745000+00:00,2026-01-31


In [15]:
ids_usati = df_moltbook_comments["post_id"].dropna().unique()

ps = (df_moltbook_posts.loc[df_moltbook_posts["id"].isin(ids_usati), ["id", "submolt"]]
      .drop_duplicates("id"))
mappa = ps.set_index("id")["submolt"]
del ps; gc.collect()

df_moltbook_comments["submolt"] = (df_moltbook_comments["post_id"]
                                   .map(mappa)
                                   .astype("category")) 

In [16]:
df_moltbook_comments.head()

,id,post_id,agent_name,agent_id,parent_id,score,created_at,dump_date,submolt
0,88660d9e-333a-4a7d-b545-07d38c3257a9,0f5bd8e5-9881-4b04-865f-2b5d60e39e86,HomeAI,329cda56-226f-4658-aecb-f40d930ab525,<NA>,0,2026-02-03 14:30:40.510000+00:00,2026-02-03,general
1,fc73fe5c-3fbd-43c7-aca3-cab2c23508a2,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,poo,d9318a43-f03a-480c-994d-e847bcf9612b,<NA>,0,2026-02-03 00:24:44.540000+00:00,2026-02-03,builds
2,e3c0997f-3ff0-4ec0-be86-3c2e74dcafd9,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,JamesBishop,21df0603-6457-4cd6-a693-fb93da4ccc44,<NA>,0,2026-02-03 00:38:37.074000+00:00,2026-02-03,builds
3,d31395b9-442f-48d0-8ec5-72112bdfd611,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,StratusBot,611761a2-d690-49ae-986f-33b3f5dccd36,<NA>,0,2026-02-03 01:45:12.210000+00:00,2026-02-03,builds
4,ab8a0ec2-ad4b-4c9c-adda-aab8441779f0,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,[deleted],b6a156b8-4cf8-40aa-927f-7e4a0c758c5b,<NA>,0,2026-02-03 08:52:58.033000+00:00,2026-02-03,builds


In [9]:
df_moltbook_posts.head()

,id,agent_id,submolt,comment_count,created_at,dump_date
0,2c7e039f-cbe0-4861-9f7e-d6f3bf04351b,606d94f2-9fde-414b-add3-d326c9ab3ccc,builds,344,2026-01-31 14:29:34.198767+00:00,2026-01-31
1,98df15b6-4f63-4782-a967-6d9aa2239cf0,37944d7b-2721-4b24-9b61-00f37bdeaa90,agents,6927,2026-01-31 12:21:30.443643+00:00,2026-01-31
2,94fc8fda-a6a9-4177-8d6b-e499adb9d675,f9658711-c03b-4c78-ae65-6cb2d33e17b7,general,48758,2026-01-31 14:00:14.933000+00:00,2026-01-31
3,75404525-5e5e-4778-ad1b-3fac43c6903d,10513ade-652d-4c8c-a410-7e686dba652c,general,31919,2026-01-31 18:11:19.470000+00:00,2026-01-31
4,c6eb531f-1ee8-428b-b1d8-41af2e9bd537,0bd24cc2-f042-4951-8400-38bd655102c6,general,5989,2026-01-31 13:12:48.745000+00:00,2026-01-31


In [17]:
SUB = ["philosophy", "crypto", "technology"]
da, a = pd.Timestamp("2026-05-01", tz="UTC"), pd.Timestamp("2026-10-01", tz="UTC")

df_moltbook_comments_small = df_moltbook_comments[df_moltbook_comments["submolt"].isin(SUB) & (df_moltbook_comments["created_at"] >= da) & (df_moltbook_comments["created_at"] < a)].copy()
df_moltbook_posts_small = df_moltbook_posts[df_moltbook_posts["submolt"].isin(SUB) & (df_moltbook_posts["created_at"] >= da) & (df_moltbook_posts["created_at"] < a)].copy()

df_moltbook_comments_small['interaction_type'] = 'comment'
df_moltbook_posts_small['interaction_type'] = 'post'

In [18]:
df_moltbook_comments_small.head()

,id,post_id,agent_name,agent_id,parent_id,score,created_at,dump_date,submolt,interaction_type
1410477,1864a3e9-11bc-413f-885d-c983bdeeb516,aab05ad4-2f28-467f-87f9-795f23dde771,vybesbot,104835f1-df5a-465e-97dc-132c2f6d774a,<NA>,0,2026-05-01 00:00:24.449000+00:00,2026-05-01,crypto,comment
1410528,56ed62b1-9ee7-4bff-bce6-1b2c8737f2a1,7031600a-8d3a-4e70-888a-9eb0de1e2777,Gordon_Gekko,fd8e5ef4-500a-4c4d-9f9c-59e5482f9b7d,<NA>,0,2026-05-01 00:03:17.580000+00:00,2026-05-01,philosophy,comment
1410553,064e1ecd-ae2d-4720-ac90-96f1db85fc9a,332b55c7-86c5-4a8f-94b7-484f65d1b2a2,trinityluthams,75f95330-01b9-4088-a6b9-3acfc5ca5d19,<NA>,0,2026-05-01 00:06:20.513000+00:00,2026-05-01,crypto,comment
1410603,e15164c0-cff7-471a-9b0b-e9b8a9932d97,de9cb959-e118-4e85-8e24-b80f51ae59d7,Sabline,ee346792-7cd5-4eba-917b-37f7de3d84c4,<NA>,0,2026-05-01 00:14:55.902000+00:00,2026-05-01,technology,comment
1410608,1830368c-435a-4f19-82f5-c8e234f69fe4,c9347efe-6736-48a3-ad72-aa0008c95a3f,defiyieldmeister,c502819a-12a0-4509-ad26-939865736452,<NA>,0,2026-05-01 00:17:47.917000+00:00,2026-05-01,crypto,comment


In [19]:
df_moltbook_posts_small.head()

,id,agent_id,submolt,comment_count,created_at,dump_date,interaction_type
2798462,afd38b1d-b1b9-4f0c-b4af-24d9083bd805,ceec6dec-b69e-4b2a-985c-71348c9438c1,philosophy,0,2026-05-01 00:01:02.365000+00:00,2026-05-01,post
2798471,b701d7f0-1da8-4635-88cd-41fd0122174f,26dccf8a-6f85-4151-895f-35a69b237de9,philosophy,0,2026-05-01 00:00:44.245000+00:00,2026-05-01,post
2798481,a5f71d55-d032-4aa1-8214-920eacff9771,923cc764-ad3f-4017-8384-4f6a3e5d504e,philosophy,0,2026-05-01 00:00:24.191000+00:00,2026-05-01,post
2798497,aab05ad4-2f28-467f-87f9-795f23dde771,f62d15ca-f65f-4dc7-adbf-0fc648dcc803,crypto,1,2026-05-01 00:00:08.969000+00:00,2026-05-01,post
2798517,7031600a-8d3a-4e70-888a-9eb0de1e2777,aa25e979-ad94-4474-88a7-44c9a0ddb813,philosophy,1,2026-05-01 00:01:45.826000+00:00,2026-05-01,post


In [20]:
df_moltbook_comments_small.to_pickle('df_moltbook_comments_small.pkl')
df_moltbook_posts_small.to_pickle('df_moltbook_posts_small.pkl')